# Phase 1 Notebook: Mention Detection
Reads text archives from data/01_input and writes only to data/10_mention_detection.

## 1) Project Setup
Resolve repository paths and make the source package importable in this notebook session.

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'data').exists() and (cur / 'speakermining' / 'src').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'speakermining' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

WindowsPath('C:/workspace/git/borgnetzwerk/speaker-mining')

## 2) Input Discovery
Build the list of archive text files that will be parsed into structured episode blocks.

In [2]:
from process.mention_detection.config import DEFAULT_PDF_TXT_INPUTS, ZDF_ARCHIVE_DIR, PHASE_DIR
from process.text_extraction.text import load_episode_blocks_from_many

input_paths = [ROOT / ZDF_ARCHIVE_DIR / name for name in DEFAULT_PDF_TXT_INPUTS]
input_paths

[WindowsPath('C:/workspace/git/borgnetzwerk/speaker-mining/data/01_input/zdf_archive/Markus Lanz_2008-2010.pdf_episodes.txt'),
 WindowsPath('C:/workspace/git/borgnetzwerk/speaker-mining/data/01_input/zdf_archive/Markus Lanz_2011-2015.pdf_episodes.txt'),
 WindowsPath('C:/workspace/git/borgnetzwerk/speaker-mining/data/01_input/zdf_archive/Markus Lanz_2016-2020.pdf_episodes.txt'),
 WindowsPath('C:/workspace/git/borgnetzwerk/speaker-mining/data/01_input/zdf_archive/Markus Lanz_2021-2024.pdf_episodes.txt')]

## 3) Episode And Season Extraction
Parse text blocks into episode records, then derive season-level aggregates from extracted episode metadata.

In [3]:
from process.mention_detection.episode import extract_episode_and_publication_rows, save_episodes
from process.mention_detection.publications import save_publications
from process.mention_detection.season import extract_season_rows, save_seasons

episode_blocks = load_episode_blocks_from_many(input_paths)
episodes_df, publications_df = extract_episode_and_publication_rows(episode_blocks)
seasons_df = extract_season_rows(episodes_df)

ep_path = save_episodes(episodes_df, ROOT / PHASE_DIR)
pu_path = save_publications(publications_df, ROOT / PHASE_DIR)
se_path = save_seasons(seasons_df, ROOT / PHASE_DIR)

print(f'episodes: {len(episodes_df)} -> {ep_path}')
print(f'publications: {len(publications_df)} -> {pu_path}')
print(f'seasons: {len(seasons_df)} -> {se_path}')

episodes: 2038 -> C:\workspace\git\borgnetzwerk\speaker-mining\data\10_mention_detection\episodes.csv
publications: 2548 -> C:\workspace\git\borgnetzwerk\speaker-mining\data\10_mention_detection\publications.csv
seasons: 17 -> C:\workspace\git\borgnetzwerk\speaker-mining\data\10_mention_detection\seasons.csv


## 4) Mention Extraction
Extract person and topic mentions from the same episode text blocks.
Institution extraction is deferred to the semantic disambiguation phase (with Wikidata context).


In [4]:
from process.mention_detection.guest import extract_person_mentions, save_person_mentions
from process.mention_detection.topic import extract_topic_mentions, save_topic_mentions

# Extract persons and topics
persons_df = extract_person_mentions(episode_blocks, episodes_df)
topics_df = extract_topic_mentions(episode_blocks, episodes_df)

pe_path = save_person_mentions(persons_df, ROOT / PHASE_DIR)
to_path = save_topic_mentions(topics_df, ROOT / PHASE_DIR)

print(f'persons: {len(persons_df)} -> {pe_path}')
print(f'topics: {len(topics_df)} -> {to_path}')


persons: 10098 -> C:\workspace\git\borgnetzwerk\speaker-mining\data\10_mention_detection\persons.csv
topics: 10713 -> C:\workspace\git\borgnetzwerk\speaker-mining\data\10_mention_detection\topics.csv


## 5) Quantitative Validation
Check key metrics and consistency signals across all extracted tables before moving to candidate generation.

### Overview

In [5]:
import pandas as pd

persons_conf = pd.to_numeric(persons_df["confidence"], errors="coerce")
topics_conf = pd.to_numeric(topics_df["confidence"], errors="coerce")

overview = pd.DataFrame(
    [
        {
            "table": "seasons",
            "rows": len(seasons_df),
            "unique_seasons": seasons_df["season_id"].nunique(),
        },
        {
            "table": "episodes",
            "rows": len(episodes_df),
            "unique_episodes": episodes_df["episode_id"].nunique(),
            "missing_publication_links": int(episodes_df["publikation_id"].isna().sum()),
            "missing_dates": int(episodes_df["publikationsdatum"].isna().sum()),
        },
        {
            "table": "publications",
            "rows": len(publications_df),
            "unique_publications": publications_df["publikation_id"].nunique(),
            "linked_episodes": publications_df["episode_id"].nunique(),
            "primary_publications": int((publications_df["is_primary"].astype(int) == 1).sum()),
        },
        {
            "table": "persons",
            "rows": len(persons_df),
            "unique_episodes": persons_df["episode_id"].nunique(),
            "unique_names": persons_df["name"].nunique(),
            "avg_confidence": round(float(persons_conf.mean()), 3),
        },
        {
            "table": "topics",
            "rows": len(topics_df),
            "unique_episodes": topics_df["episode_id"].nunique(),
            "unique_topics": topics_df["topic"].nunique(),
            "avg_confidence": round(float(topics_conf.mean()), 3),
        },
    ]
).fillna("")

season_dates = seasons_df.assign(
    start=pd.to_datetime(seasons_df["start_time"], errors="coerce", dayfirst=True),
    end=pd.to_datetime(seasons_df["end_time"], errors="coerce", dayfirst=True),
)

season_summary = {
    "season_count": len(seasons_df),
    "min_episodes_per_season": int(seasons_df["episode_count"].min()),
    "median_episodes_per_season": float(seasons_df["episode_count"].median()),
    "max_episodes_per_season": int(seasons_df["episode_count"].max()),
    "season_start": season_dates["start"].min().date().isoformat(),
    "season_end": season_dates["end"].max().date().isoformat(),
}

display(overview)
season_summary

,table,rows,unique_seasons,unique_episodes,missing_publication_links,missing_dates,unique_publications,linked_episodes,primary_publications,unique_names,avg_confidence,unique_topics
0,seasons,17,17.0,,,,,,,,,
1,episodes,2038,,2036.0,0.0,0.0,,,,,,
2,publications,2548,,,,,2542.0,2034.0,2036.0,,,
3,persons,10098,,1992.0,,,,,,4885.0,0.915,
4,topics,10713,,1874.0,,,,,,,0.776,10503.0


{'season_count': 17,
 'min_episodes_per_season': 29,
 'median_episodes_per_season': 129.0,
 'max_episodes_per_season': 142,
 'season_start': '2008-06-03',
 'season_end': '2024-12-18'}

### Episodes

In [6]:
episode_dates = pd.to_datetime(episodes_df["publikationsdatum"], errors="coerce", dayfirst=True)

episodes_per_season = (
    episodes_df.groupby("season", dropna=False)["episode_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .rename("episodes")
)

episode_validation = {
    "episodes_total": len(episodes_df),
    "unique_episode_ids": episodes_df["episode_id"].nunique(),
    "duplicate_episode_ids": int(episodes_df["episode_id"].duplicated().sum()),
    "date_coverage_start": episode_dates.min().date().isoformat(),
    "date_coverage_end": episode_dates.max().date().isoformat(),
    "missing_season": int(episodes_df["season"].isna().sum()),
    "missing_staffel": int(episodes_df["staffel"].isna().sum()),
    "missing_folge": int(episodes_df["folge"].isna().sum()),
    "missing_folgennr": int(episodes_df["folgennr"].isna().sum()),
}

publications_per_episode = publications_df.groupby("episode_id")["publikation_id"].nunique()
publication_link_stats = {
    "episodes_with_publication_rows": int(publications_per_episode.index.nunique()),
    "episodes_without_publication_rows": int(len(episodes_df) - publications_per_episode.index.nunique()),
    "publications_per_episode_min": int(publications_per_episode.min()),
    "publications_per_episode_median": float(publications_per_episode.median()),
    "publications_per_episode_max": int(publications_per_episode.max()),
}

display(episodes_df)
episode_validation, publication_link_stats, episodes_per_season

,episode_id,sendungstitel,publikation_id,publikationsdatum,dauer,archivnummer,prod_nr_beitrag,zeit_tc_start,zeit_tc_end,season,staffel,folge,folgennr,infos
178,ep_a371a3777018,Markus Lanz 03.06.2008,pb_e81cb7939cdb,03.06.2008,69'54,4010956801,00529/00175,,,"Markus Lanz, Staffel 1",,,,00:01:48 - 01:11:14 069'26 Interview Markus LA...
177,ep_36337a9dfb46,Markus Lanz 04.06.2008,pb_0b8fd4b20dc9,04.06.2008,59'51,4010941101,00529/00176,,,"Markus Lanz, Staffel 1",,,,23:17:10 - 00:16:35 059'25 Interview Markus LA...
176,ep_145cdbbd9501,Markus Lanz 05.06.2008,pb_76cf822d1aa2,05.06.2008,60'12,4010950101,00529/00177,,,"Markus Lanz, Staffel 1",,,,23:16:15 - 00:15:52 059'37 Interview Markus LA...
175,ep_75f5025ddda9,Markus Lanz 10.06.2008,pb_e23dfc1fd5bb,10.06.2008,62'46,4010999301,00529/00178,,,"Markus Lanz, Staffel 1",,,,00:01:40 - 01:03:58 062'18 Interview Markus LA...
174,ep_e102c24f59df,Markus Lanz 18.06.2008,pb_fa69a427f473,18.06.2008,61'00,4011051301,00529/00179,,,"Markus Lanz, Staffel 1",,,,23:47:56 - 00:48:24 060'28 Interview Markus LA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1487,ep_09fb1d4a4a8b,Markus Lanz 11.12.2024,pb_0a8791d9f754,11.12.2024,48'08,4122397701,00529/13993,00:02:44,00:50:36,"Markus Lanz, Staffel 17",17,134,2022,00:02:44 - 00:50:36 047'52 O-Ton Interview Mar...
1486,ep_a2ec1ab7a2ea,Markus Lanz 12.12.2024,pb_d6109046b4c1,12.12.2024,75'26,4122444201,00529/14489,23:18:42,00:33:29,"Markus Lanz, Staffel 17",17,135,2023,23:18:42 - 00:33:29 074'47 O-Ton Interview Mar...
1485,ep_88535b1c4e76,Markus Lanz 17.12.2024,pb_816fcacd7f1c,17.12.2024,75'38,4122615901,00529/14490,22:45:19,00:00:58,"Markus Lanz, Staffel 17",17,136,2024,22:45:19 - 00:00:58 075'39 O-Ton Interview Mar...
1484,ep_a1b1ceb3f697,Markus Lanz 18.12.2024,pb_204bda27e721,18.12.2024,75'03,4122661601,,23:14:49,00:29:52,"Markus Lanz, Staffel 17",17,137,2025,23:14:49 - 00:29:52 075'03 O-Ton Interview Mar...


({'episodes_total': 2038,
  'unique_episode_ids': 2036,
  'duplicate_episode_ids': 2,
  'date_coverage_start': '2008-06-03',
  'date_coverage_end': '2024-12-19',
  'missing_season': 0,
  'missing_staffel': 0,
  'missing_folge': 0,
  'missing_folgennr': 0},
 {'episodes_with_publication_rows': 2034,
  'episodes_without_publication_rows': 4,
  'publications_per_episode_min': 1,
  'publications_per_episode_median': 1.0,
  'publications_per_episode_max': 4},
 season
 Markus Lanz, Staffel 14    142
 Markus Lanz, Staffel 13    137
 Markus Lanz, Staffel 5     137
 Markus Lanz, Staffel 17    137
 Markus Lanz, Staffel 16    137
 Markus Lanz, Staffel 12    135
 Markus Lanz, Staffel 15    135
 Markus Lanz, Staffel 7     129
 Markus Lanz, Staffel 9     129
 Markus Lanz, Staffel 11    128
 Name: episodes, dtype: int64)

### Publications

In [7]:
pub_dates = pd.to_datetime(publications_df["date"], errors="coerce", dayfirst=True)

publications_validation = {
    "rows_total": len(publications_df),
    "unique_publication_ids": publications_df["publikation_id"].nunique(),
    "duplicate_publication_ids": int(publications_df["publikation_id"].duplicated().sum()),
    "linked_episode_count": publications_df["episode_id"].nunique(),
    "primary_rows": int((publications_df["is_primary"].astype(int) == 1).sum()),
    "non_primary_rows": int((publications_df["is_primary"].astype(int) == 0).sum()),
    "date_start": pub_dates.min().date().isoformat(),
    "date_end": pub_dates.max().date().isoformat(),
    "missing_time": int(publications_df["time"].isna().sum()),
    "missing_duration": int(publications_df["duration"].isna().sum()),
}

program_distribution = (
    publications_df["program"]
    .fillna("<missing>")
    .value_counts()
    .rename_axis("program")
    .to_frame("rows")
)

display(publications_df)
publications_validation, program_distribution

,publikation_id,episode_id,publication_index,date,time,duration,program,prod_nr_sendung,prod_nr_secondary,is_primary,raw_line
312,pb_e81cb7939cdb,ep_a371a3777018,1,03.06.2008,22:44:42,69'54,ZDF,00529/00175,,1,03.06.2008 22:44:42 69'54 00529/00175 ZDF
313,pb_12f66ecd45ea,ep_a371a3777018,2,04.06.2008,02:35:35,69'52,ZDF,00529/50117,,0,04.06.2008 02:35:35 69'52 00529/50117 ZDF
314,pb_d9e6d9bab453,ep_a371a3777018,3,07.06.2008,17:03:31,66'58,ZDFdokukanal,00742/08091,,0,07.06.2008 17:03:31 66'58 00742/08091 ZDFdokuk...
309,pb_0b8fd4b20dc9,ep_36337a9dfb46,1,04.06.2008,23:16:55,59'51,ZDF,00529/00176,,1,04.06.2008 23:16:55 59'51 00529/00176 ZDF
310,pb_dd749526b8b7,ep_36337a9dfb46,2,05.06.2008,02:00:51,59'50,ZDF,00529/50118,,0,05.06.2008 02:00:51 59'50 00529/50118 ZDF
...,...,...,...,...,...,...,...,...,...,...,...
1995,pb_0a8791d9f754,ep_09fb1d4a4a8b,1,11.12.2024,00:02:43,48'08,ZDF,00529/13993,,1,11.12.2024 00:02:43 48'08 00529/13993 ZDF
1994,pb_d6109046b4c1,ep_a2ec1ab7a2ea,1,12.12.2024,23:18:18,75'26,ZDF,00529/14489,,1,12.12.2024 23:18:18 75'26 00529/14489 ZDF
1993,pb_816fcacd7f1c,ep_88535b1c4e76,1,17.12.2024,22:45:19,75'38,ZDF,00529/14490,,1,17.12.2024 22:45:19 75'38 00529/14490 ZDF
1992,pb_204bda27e721,ep_a1b1ceb3f697,1,18.12.2024,23:14:49,75'03,ZDF,00529/14491,,1,18.12.2024 23:14:49 75'03 00529/14491 ZDF


({'rows_total': 2548,
  'unique_publication_ids': 2542,
  'duplicate_publication_ids': 6,
  'linked_episode_count': 2034,
  'primary_rows': 2036,
  'non_primary_rows': 512,
  'date_start': '2008-06-03',
  'date_end': '2024-12-19',
  'missing_time': 0,
  'missing_duration': 0},
               rows
 program           
 ZDF           2162
 3sat           316
 ZDFneo          29
 ZDFinfo         21
 ZDFdokukanal    18
 ZDFkultur        1
 Phoenix          1)

### Persons

In [8]:
persons_conf = pd.to_numeric(persons_df["confidence"], errors="coerce")
persons_per_episode = persons_df.groupby("episode_id")["mention_id"].nunique()

person_validation = {
    "mention_rows": len(persons_df),
    "unique_mention_ids": persons_df["mention_id"].nunique(),
    "duplicate_mention_ids": int(persons_df["mention_id"].duplicated().sum()),
    "episodes_with_person_mentions": int(persons_df["episode_id"].nunique()),
    "unique_person_names": int(persons_df["name"].nunique()),
    "avg_mentions_per_episode": round(float(persons_per_episode.mean()), 2),
    "median_mentions_per_episode": float(persons_per_episode.median()),
    "max_mentions_in_one_episode": int(persons_per_episode.max()),
    "avg_confidence": round(float(persons_conf.mean()), 3),
    "p10_confidence": round(float(persons_conf.quantile(0.10)), 3),
    "p50_confidence": round(float(persons_conf.quantile(0.50)), 3),
    "p90_confidence": round(float(persons_conf.quantile(0.90)), 3),
    "missing_description": int(persons_df["beschreibung"].isna().sum()),
    "invalid_confidence_rows": int(persons_conf.isna().sum()),
}

top_persons = (
    persons_df["name"]
    .value_counts()
    .head(10)
    .rename_axis("name")
    .to_frame("mentions")
)

person_rule_distribution = (
    persons_df["parsing_rule"]
    .value_counts()
    .rename_axis("parsing_rule")
    .to_frame("rows")
)

display(persons_df)
person_validation, top_persons, person_rule_distribution

,mention_id,episode_id,name,beschreibung,source_text,source_context,parsing_rule,confidence,confidence_note
1,pm_13aa6f5ec26b,ep_a371a3777018,Franjo Pooth,,über die Insolvenzaffäre von ihrem Ehemann Fra...,Verona POOTH (Werbeikone) über die Insolvenzaf...,name_without_local_parenthetical,0.45,name appears in multi-name chain; description ...
6,pm_29eaeb7e7965,ep_a371a3777018,Georg PIEPER,Traumaexperte,und seine Ehefrau Monika BAUCH und Georg PIEPE...,Verona POOTH (Werbeikone) über die Insolvenzaf...,last_name_parenthetical,0.82,description assigned to nearest name before pa...
2,pm_68f5f9919a8e,ep_a371a3777018,Jelena WAHLER,"Gründerin Elitekindergarten ""Little Giants""",über die Insolvenzaffäre von ihrem Ehemann Fra...,Verona POOTH (Werbeikone) über die Insolvenzaf...,last_name_parenthetical,0.82,description assigned to nearest name before pa...
5,pm_ef26d254278b,ep_a371a3777018,Monika BAUCH,,und seine Ehefrau Monika BAUCH und Georg PIEPE...,Verona POOTH (Werbeikone) über die Insolvenzaf...,name_without_local_parenthetical,0.45,name appears in multi-name chain; description ...
3,pm_81204e736b4d,ep_a371a3777018,Peter WAHLER,,"und ihr Ehemann Peter WAHLER, Udo BAUCH (Überl...",Verona POOTH (Werbeikone) über die Insolvenzaf...,name_without_local_parenthetical,0.45,name appears in multi-name chain; description ...
...,...,...,...,...,...,...,...,...,...
10092,pm_afd9b59e41b1,ep_5105776167e0,Niclas FÜLLKRUG,"Fußballprofi, Nationalspieler Deutschland, Zus...","Niclas FÜLLKRUG (Fußballprofi, Nationalspieler...",den Studiogästen BMWK Robert HABECK (Bündnis 9...,single_parenthetical,0.95,single name directly tied to parenthetical des...
10090,pm_07a08304e1f8,ep_5105776167e0,Robert ANDRICH,"Fußballprofi, Bayer 04 Leverkusen, Zuschaltung...","Robert ANDRICH (Fußballprofi, Bayer 04 Leverku...",den Studiogästen BMWK Robert HABECK (Bündnis 9...,single_parenthetical,0.95,single name directly tied to parenthetical des...
10084,pm_963786359058,ep_5105776167e0,Robert HABECK,Bündnis 90 / Die Grünen,den Studiogästen BMWK Robert HABECK (Bündnis 9...,den Studiogästen BMWK Robert HABECK (Bündnis 9...,single_parenthetical,0.95,single name directly tied to parenthetical des...
10096,pm_f70ef1b12e9d,ep_5105776167e0,Ryyan ALSHEBL,"Geflüchteter aus Syrien, Bgm Gemeinde Ostelshe...","Ryyan ALSHEBL (Geflüchteter aus Syrien, Bgm Ge...",den Studiogästen BMWK Robert HABECK (Bündnis 9...,single_parenthetical,0.95,single name directly tied to parenthetical des...


({'mention_rows': 10098,
  'unique_mention_ids': 10098,
  'duplicate_mention_ids': 0,
  'episodes_with_person_mentions': 1992,
  'unique_person_names': 4885,
  'avg_mentions_per_episode': 5.07,
  'median_mentions_per_episode': 5.0,
  'max_mentions_in_one_episode': 21,
  'avg_confidence': 0.915,
  'p10_confidence': 0.82,
  'p50_confidence': 0.95,
  'p90_confidence': 0.95,
  'missing_description': 0,
  'invalid_confidence_rows': 0},
                   mentions
 name                      
 Elmar THEVEßEN          78
 Robin ALEXANDER         66
 Karl LAUTERBACH         53
 Eva QUADBECK            40
 Kristina DUNZ           37
 Michael SPRENG          33
 Kevin KÜHNERT           31
 Wolfgang KUBICKI        30
 Markus SÖDER            30
 Hajo SCHUMACHER         30,
                                   rows
 parsing_rule                          
 single_parenthetical              9034
 name_without_local_parenthetical   742
 last_name_parenthetical            271
 group_parenthetical        

### Topics

In [9]:
topics_conf = pd.to_numeric(topics_df["confidence"], errors="coerce")
topics_per_episode = topics_df.groupby("episode_id")["mention_id"].nunique()

topic_validation = {
    "mention_rows": len(topics_df),
    "unique_mention_ids": topics_df["mention_id"].nunique(),
    "duplicate_mention_ids": int(topics_df["mention_id"].duplicated().sum()),
    "episodes_with_topic_mentions": int(topics_df["episode_id"].nunique()),
    "unique_topic_labels": int(topics_df["topic"].nunique()),
    "avg_topics_per_episode": round(float(topics_per_episode.mean()), 2),
    "median_topics_per_episode": float(topics_per_episode.median()),
    "max_topics_in_one_episode": int(topics_per_episode.max()),
    "avg_confidence": round(float(topics_conf.mean()), 3),
    "p10_confidence": round(float(topics_conf.quantile(0.10)), 3),
    "p50_confidence": round(float(topics_conf.quantile(0.50)), 3),
    "p90_confidence": round(float(topics_conf.quantile(0.90)), 3),
    "invalid_confidence_rows": int(topics_conf.isna().sum()),
}

top_topics = (
    topics_df["topic"]
    .value_counts()
    .head(10)
    .rename_axis("topic")
    .to_frame("mentions")
)

topic_rule_distribution = (
    topics_df["parsing_rule"]
    .value_counts()
    .rename_axis("parsing_rule")
    .to_frame("rows")
)

display(topics_df)
topic_validation, top_topics, topic_rule_distribution

,mention_id,episode_id,topic,source_text,source_context,parsing_rule,confidence,confidence_note
0,tm_6fc11a604d1a,ep_a371a3777018,Insolvenzaffäre von ihrem Ehemann Franjo Pooth,über die Insolvenzaffäre von ihrem Ehemann Fra...,Interview Markus LANZ mit Verona POOTH (Werbei...,cue_based_clause,0.72,topic inferred from über/zu cue in semicolon c...
1,tm_4996c7bec51c,ep_adf92e4548b0,BORG (Hai-Schützer) und Folkart SCHWEIZER (wur...,ZUR BORG (Hai-Schützer) und Folkart SCHWEIZER ...,Interview Markus LANZ mit den Studiogästen Lot...,cue_based_clause,0.72,topic inferred from über/zu cue in semicolon c...
2,tm_32664f1680c7,ep_706817edfa58,Gesundheitsmythen und Schönheitsoperationen,zu Gesundheitsmythen und Schönheitsoperationen,Interview Markus LANZ mit den Studiogästen Wer...,cue_based_clause,0.72,topic inferred from über/zu cue in semicolon c...
3,tm_0f91e6ea3cbe,ep_8f5659760693,"seiner Tochter, und der Entbindung seiner im K...","zu seiner Tochter, und der Entbindung seiner i...",Interview Markus LANZ mit den Studiogästen Gis...,cue_based_clause,0.72,topic inferred from über/zu cue in semicolon c...
4,tm_1cbe43faae48,ep_8e128514d9e6,Hamburg),zu Hamburg),Interview Markus LANZ mit den Studiogästen And...,cue_based_clause,0.72,topic inferred from über/zu cue in semicolon c...
...,...,...,...,...,...,...,...,...
10705,tm_db5d0a6c6369,ep_88535b1c4e76,insbesondere Finanzpolitik,insbesondere Finanzpolitik,Interview Markus LANZ mit den Studiogästen Tho...,inline_label_comma_split,0.62,topic from inline Themen/Schwerpunktthemen lis...
10709,tm_3a6af9f52023,ep_a1b1ceb3f697,das Leistungsniveau der Schüler und die Ergebn...,: das Leistungsniveau der Schüler und die Erge...,Interview Markus LANZ mit den Studiogästen Sil...,labelled_topic_list,0.93,topic from explicit Thema/Themen/Schwerpunktth...
10711,tm_2e8312132620,ep_a1b1ceb3f697,die Erziehungsaufgabe der Schulen,die Erziehungsaufgabe der Schulen,Interview Markus LANZ mit den Studiogästen Sil...,labelled_topic_list,0.93,topic from explicit Thema/Themen/Schwerpunktth...
10712,tm_a4c836534da7,ep_a1b1ceb3f697,die Herausforderungen an Brennpunktschulen und...,die Herausforderungen an Brennpunktschulen und...,Interview Markus LANZ mit den Studiogästen Sil...,labelled_topic_list,0.93,topic from explicit Thema/Themen/Schwerpunktth...


({'mention_rows': 10713,
  'unique_mention_ids': 10713,
  'duplicate_mention_ids': 0,
  'episodes_with_topic_mentions': 1874,
  'unique_topic_labels': 10503,
  'avg_topics_per_episode': 5.72,
  'median_topics_per_episode': 5.0,
  'max_topics_in_one_episode': 36,
  'avg_confidence': 0.776,
  'p10_confidence': 0.56,
  'p50_confidence': 0.76,
  'p90_confidence': 0.93,
  'invalid_confidence_rows': 0},
                                       mentions
 topic                                         
 der Krieg in der Ukraine                     8
 Schönheitsoperationen                        6
 Griechenland-Krise                           6
 Älterwerden                                  5
 haben                                        5
 Eurovision Songcontest                       5
 Flüchtlingsdrama in Europa                   5
 Migrationspolitik                            5
 die Waffenlieferungen an die Ukraine         5
 sein                                         4,
                      